# 18. Color Defect Shape 상호작용 분석

단일 변수 평균이 아니라 조합별 성능 하락을 확인합니다.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-1장/ch2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2장/ch2_utils.py"))
        + list(Path.cwd().glob("**/ch2_utils.py"))
    )
    NOTEBOOK_DIR = matches[0].parent if matches else Path("Deeplearning") / "Vision 응용" / "2-1장"
sys.path.append(str(NOTEBOOK_DIR))

from ch2_utils import *

paths = find_paths()
set_korean_font()
set_seed(7)
samples = load_samples(paths.data_root)
paths

In [ ]:
run_dir = paths.runs_root / "baseline_segformer_b0"
sample_metrics, group_metrics, class_metrics = load_run_metrics(run_dir)

## 18-1. Color x Defect interaction

In [ ]:
cd = plot_metric_heatmap(
    sample_metrics,
    row="color_group",
    col="defect_type",
    metric="target_dice",
    title="Color x Defect interaction",
    out_path=paths.runs_root / "18_color_defect_interaction.png",
)

## 18-2. Shape x Defect interaction

In [ ]:
sd = plot_metric_heatmap(
    sample_metrics,
    row="shape_group",
    col="defect_type",
    metric="target_dice",
    title="Shape x Defect interaction",
    out_path=paths.runs_root / "18_shape_defect_interaction.png",
)

## 18-3. Worst combination table

In [ ]:
combo = (
    sample_metrics.groupby(["color_group", "shape_group", "defect_type"])["target_dice"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values("mean")
)
combo.to_csv(paths.runs_root / "interaction_effects_b0.csv", index=False, encoding="utf-8-sig")
display(combo.head(10))
display(combo.tail(10))

## 18-4. 상호작용 결론

In [ ]:
worst = combo.iloc[0]
best = combo.iloc[-1]
print(
    f"결론: color-shape-defect 조합 중 최저 성능은 "
    f"{worst['color_group']} / {worst['shape_group']} / {worst['defect_type']} (Dice={worst['mean']:.3f})입니다. "
    f"최고 조합 {best['color_group']} / {best['shape_group']} / {best['defect_type']} (Dice={best['mean']:.3f})와의 차이는 "
    f"{best['mean'] - worst['mean']:.3f}입니다."
)